# Validated GenAI Analysis of Public Comments on the FTC's Personalized-Pricing Policy

## 1. Business understanding

**Context.** In August 2026 the Federal Trade Commission (FTC) proposed an enforcement policy on *personalized pricing*: businesses using a consumer's personal data to estimate what that person will pay, and setting their price accordingly. The proposal does not ban the practice. It says businesses may break the law if they fail to clearly disclose that a price is personalized, why, and what data was used. The public comment period (docket FTC-2026-1057) closed on September 25, 2026, with 3,706 public comments.

**The business problem.** Regulators, law firms and public-affairs teams must read and summarize thousands of comments like these by hand. GenAI could do this faster and more cheaply, but a misread comment misrepresents the public record. Before anyone relies on it, the AI's accuracy has to be measured against careful human judgment.

**Research questions.**

*Workflow (for organizations adopting GenAI):*
1. How accurately does an LLM classify comments compared with human coders?
2. What does it cost per comment compared with human analysts?
3. Where does it fail?
4. What controls make it safe to use?

*Findings (for companies that use personalized pricing):*

5. What do individual commenters object to: the price difference, the use of their data, or the lack of disclosure?
6. What do they want the FTC to do?

**Scope.** This study analyzes comments from **individuals**. Submissions from organizations (companies, trade groups, advocacy groups) are mostly long attached documents that need a different method; comparing them with individuals is left for future work.

**Approach.** Understand → Use → Evaluate → Govern. Findings are reported only for labels the AI has been shown to classify reliably.

## Setup

In [ ]:
# Loads the libraries and defines the settings and file paths
# used in the rest of the notebook. Every setting that affects
# results is defined here, so it only needs changing in one place.

# Standard library
import os                      # environment variables (e.g., the API key)
import re                      # regular expressions, for cleaning text
import html                    # decodes HTML codes such as &rsquo; into normal characters
import json                    # reads and writes the LLM's structured (JSON) output
from pathlib import Path       # builds file paths that work on Windows and Mac

# Data analysis and plotting
import numpy as np             # numerical operations
import pandas as pd            # loading and manipulating the comment data
import matplotlib.pyplot as plt  # charts
import seaborn as sns          # nicer charts

# Reproducibility settings
# The same random seed was used to draw the comment sample from the API,
# so anyone rerunning the notebook gets exactly the same results.
SEED = 20261027

# Total public comments on docket FTC-2026-1057, as reported by the
# Regulations.gov API. Used to describe how much of the docket we analyze.
DOCKET_TOTAL = 3706

# File paths
# The notebook lives in Notebooks/, but the data is at the top of the repo.
# Find the repo's top folder by walking up until we find the Data folder,
# so the paths work no matter where the notebook is run from.
ROOT = Path.cwd()
while not (ROOT / "Data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Input: individual comments pulled from the Regulations.gov API.
DATA_FILE = ROOT / "Data" / "individuals_sentiment_data.csv"

# Output: folder for saved charts, at the top of the repo; created if needed.
FIG = ROOT / "outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# Display settings
# Show up to 120 characters of each comment when printing tables.
pd.set_option("display.max_colwidth", 120)

# Check the input file exists before going further
# Stops the notebook with a clear message if the file is missing or misnamed,
# instead of failing later with a confusing error.
assert DATA_FILE.exists(), f"Missing {DATA_FILE}: check the folder and file name"
print("Setup complete. Data file found:", DATA_FILE)

Setup complete. Data file found: c:\Users\ADMIN\Documents\Daniels College of Business\Fall 2026\Analytics & AI\ftc-pricing-comments-genai\Data\individuals_sentiment_data.csv


## 2. Data understanding

**Source.** Public comments on docket FTC-2026-1057, retrieved from the official Regulations.gov v4 API. All comment IDs on the docket were shuffled with a fixed seed (20261027) and fetched in that order, so the comments retrieved form a simple random sample of the docket.

**Unit of observation.** One public comment submitted by an individual.

**Privacy.** Commenters' names were excluded at download. Only the comment text, posting date and, where given, the U.S. state are kept.

In [9]:
# LOAD THE DATA AND TAKE A FIRST LOOK
# Before cleaning anything, check what the file contains:
# its size, its columns, their types and any missing values.

# Load the comments
df_raw = pd.read_csv(DATA_FILE)

# Size of the dataset
# Each row should be one comment.
print(f"Rows (comments): {len(df_raw):,}")
print(f"Columns: {df_raw.shape[1]}")
print(f"Share of the full docket: {len(df_raw) / DOCKET_TOTAL:.1%} of {DOCKET_TOTAL:,} comments\n")

#  Columns and their data types
# Confirms each column was read as expected (text, numbers, dates).
print("Columns and types:")
print(df_raw.dtypes, "\n")

# Missing values per column
# Some missing values are expected (e.g., not every commenter gives a state).
print("Missing values per column:")
print(df_raw.isna().sum(), "\n")

# A few example rows
# Shows what the raw comment text looks like before cleaning.
df_raw.head()

Rows (comments): 1,566
Columns: 10
Share of the full docket: 42.3% of 3,706 comments

Columns and types:
comment_id           str
stratum              str
posted_date          str
organization     float64
state                str
withdrawn           bool
n_attachments      int64
comment_text         str
n_words            int64
shuffle_rank       int64
dtype: object 

Missing values per column:
comment_id          0
stratum             0
posted_date         0
organization     1566
state             554
withdrawn           0
n_attachments       0
comment_text        0
n_words             0
shuffle_rank        0
dtype: int64 



,comment_id,stratum,posted_date,organization,state,withdrawn,n_attachments,comment_text,n_words,shuffle_rank
0,FTC-2026-1057-0118,A_individual,2026-08-31T04:00:00Z,NaN,MA,False,0,"Dear Sir or Madam,\r\n\r\nSurveillance pricing is a threat to consumers, and the practice must be stopped altogether...",112,0
1,FTC-2026-1057-2221,A_individual,2026-09-24T04:00:00Z,NaN,FL,False,0,"Company personal pricing should not be allowed using AI, financial data, web-based information, or other individual ...",17,1
2,FTC-2026-1057-2723,A_individual,2026-09-24T04:00:00Z,NaN,NaN,False,0,Absolutely preposterous idea to set prices based on personal information!!,10,2
3,FTC-2026-1057-1366,A_individual,2026-09-23T04:00:00Z,NaN,NaN,False,0,Personalized pricing is just unfair. It pits the rich vs the middle class vs the poor. Demand that companies abandon...,22,3
4,FTC-2026-1057-3459,A_individual,2026-09-25T04:00:00Z,NaN,NaN,False,0,Life is difficult enough to have to worry about whether your shopping habits are costing you money! This is delibera...,46,4


### what the data contains

- **1,566 individual comments**, 42.3% of the 3,706 comments on the docket. They come from the first 1,630 comments in the fixed random order (the other 64 were organizations, excluded from this study), so they form a random sample of individual commenters.
- **10 columns**: a comment ID, the comment text, a word count, the posting date, the commenter's U.S. state, the position in the random order (`shuffle_rank`), and four columns describing the submission type (`stratum`, `organization`, `withdrawn`, `n_attachments`).
- **Missing values are expected, not errors.** `organization` is empty for every row because these are individuals. `state` is missing for 554 comments (35%) because giving a state is optional on Regulations.gov.
- **No missing comment text, IDs or dates**, so every row can be analyzed.
- The four submission-type columns should be the same for every row (individual, no organization, not withdrawn, no attachments). We verify this in cleaning and then drop them.

## 3. Data cleaning

Each cleaning step is checked with an `assert`, so the notebook stops with a clear message if the data is not what we expect.

In [ ]:
# Turns the raw file into an analysis-ready table and checks
# every assumption we rely on later.

df = df_raw.copy()   # keep the raw data untouched for reference

# 1. Confirm every row is an individual comment
# These columns describe submission type. For individuals they should
# be identical on every row; if so, they carry no information.
assert (df["stratum"] == "A_individual").all(), "Non-individual rows found"
assert df["organization"].isna().all(), "Some rows name an organization"
assert (~df["withdrawn"]).all(), "Some comments were withdrawn"
assert (df["n_attachments"] == 0).all(), "Some comments have attachments"
df = df.drop(columns=["stratum", "organization", "withdrawn", "n_attachments"])

# 2. Clean the comment text
# Regulations.gov stores comments as HTML. Turn line-break tags into
# real line breaks, remove any other tags, decode codes like &rsquo;
# into normal characters, and collapse repeated spaces.
def clean_text(t):
    t = re.sub(r"(?i)<br\s*/?>", "\n", str(t))
    t = re.sub(r"<[^>]+>", " ", t)
    t = html.unescape(t)
    return re.sub(r"[ \t]+", " ", t).strip()

df["text"] = df["comment_text"].map(clean_text)
df = df.drop(columns=["comment_text"])

# 3. Recount words on the cleaned text
df["n_words"] = df["text"].str.split().str.len()

# 4. Convert the posting date from text to a real date
df["posted_date"] = pd.to_datetime(df["posted_date"]).dt.date

# 5. Flag (do not delete) very short comments
# Comments under 5 words often carry little meaning ("Stop this.").
# We keep them but flag them, so later steps can report their effect.
df["is_short"] = df["n_words"] < 5

# 6. Checks on the cleaned data
assert df["comment_id"].is_unique, "Duplicate comment IDs"
assert df["comment_id"].str.match(r"^FTC-2026-1057-\d{4}$").all(), "Unexpected ID format"
assert (df["text"].str.len() > 0).all(), "Empty comment text after cleaning"
assert not df["text"].str.contains(r"&[a-z]+;|<br", regex=True).any(), "HTML left in text"
assert not any(c in df.columns for c in ["first_name", "last_name", "firstName", "lastName"]), \
    "Personal names present (privacy rule)"

# 7. Sampling check
# Comments were fetched in a fixed random order (seed 20261027), and
# shuffle_rank records each comment's position in that order. Each
# position should appear once and fall within the docket's range.
assert df["shuffle_rank"].is_unique, "Two comments share a position in the random order"
assert df["shuffle_rank"].between(0, DOCKET_TOTAL - 1).all(), "Position outside the docket's range"

print(f"Clean dataset: {len(df):,} comments, columns: {list(df.columns)}")
print(f"Very short comments (<5 words): {df['is_short'].sum()} ({df['is_short'].mean():.1%})")
df.head()

Clean dataset: 1,566 comments, columns: ['comment_id', 'posted_date', 'state', 'n_words', 'shuffle_rank', 'text', 'is_short']
Very short comments (<5 words): 56 (3.6%)


,comment_id,posted_date,state,n_words,shuffle_rank,text,is_short
0,FTC-2026-1057-0118,2026-08-31,MA,112,0,"Dear Sir or Madam,\r\n\r\nSurveillance pricing is a threat to consumers, and the practice must be stopped altogether...",False
1,FTC-2026-1057-2221,2026-09-24,FL,17,1,"Company personal pricing should not be allowed using AI, financial data, web-based information, or other individual ...",False
2,FTC-2026-1057-2723,2026-09-24,NaN,10,2,Absolutely preposterous idea to set prices based on personal information!!,False
3,FTC-2026-1057-1366,2026-09-23,NaN,22,3,Personalized pricing is just unfair. It pits the rich vs the middle class vs the poor. Demand that companies abandon...,False
4,FTC-2026-1057-3459,2026-09-25,NaN,46,4,Life is difficult enough to have to worry about whether your shopping habits are costing you money! This is delibera...,False
